In [1]:
import json
import html
from pathlib import Path

# -------------------------
# INPUT / OUTPUT
# -------------------------
INPUT_JSONL = "Dataset/VIOLIN_12-10-2025/interim/exact_matches/matched_exact_abstracts.jsonl"
OUTPUT_HTML = "Dataset/VIOLIN_12-10-2025/interim/exact_matches/green_highlight_preview.html"

# -------------------------
# HTML helpers
# -------------------------
HTML_HEADER = """
<html>
<head>
<meta charset="utf-8"/>
<style>
body {
    font-family: Arial, sans-serif;
    line-height: 1.6;
    margin: 30px;
}
.abstract {
    margin-bottom: 40px;
}
.meta {
    color: #555;
    font-size: 0.9em;
    margin-bottom: 6px;
}
</style>
</head>
<body>
<h1>Adjuvant Dictionary Matches (Exact + Expanded)</h1>
"""

HTML_FOOTER = "</body></html>"

# -------------------------
# Gradient highlight helpers
# -------------------------
def gradient_for_level(level: int, max_level: int) -> str:
    """
    Map overlap depth to a green gradient.
    level: 1..max_level
    """
    if max_level <= 1:
        return "linear-gradient(90deg, #d8fbd8, #7cf27c)"
    t = (level - 1) / (max_level - 1)
    # light green -> darker green
    r = int(216 + (72 - 216) * t)
    g = int(251 + (186 - 251) * t)
    b = int(216 + (96 - 216) * t)
    return f"linear-gradient(90deg, rgb({r},{g},{b}), rgb({max(0,r-20)},{max(0,g-40)},{max(0,b-40)}))"

def highlight_text_gradient(text, spans):
    """
    spans: list of (start, end)
    Overlap-aware gradient highlighting.
    """
    if not spans:
        return html.escape(text)

    n = len(text)
    depth = [0] * n
    for start, end in spans:
        start = max(0, min(n, start))
        end = max(0, min(n, end))
        for i in range(start, end):
            depth[i] += 1

    max_level = max(depth) if depth else 0

    out = []
    i = 0
    while i < n:
        if depth[i] == 0:
            j = i + 1
            while j < n and depth[j] == 0:
                j += 1
            out.append(html.escape(text[i:j]))
            i = j
        else:
            level = depth[i]
            j = i + 1
            while j < n and depth[j] == level:
                j += 1
            color = gradient_for_level(level, max_level)
            out.append(
                f"<span style='background:{color}; padding:1px 2px; border-radius:3px;'>"
                f"{html.escape(text[i:j])}</span>"
            )
            i = j
    return "".join(out)

# -------------------------
# Build HTML
# -------------------------
html_blocks = [HTML_HEADER]

with open(INPUT_JSONL, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        abstract = record.get("abstract", "")
        title = record.get("title", "")
        pmid = record.get("pmid")

        matches = record.get("matches", [])
        if not abstract or not matches:
            continue

        #####################################
        from collections import defaultdict

        vo_map = defaultdict(set)
        for m in matches:
            vo_map[m["adjuvant_vo_id"]].add(m["matched_text"])

        vo_lines = [
            f"{vo}: {', '.join(sorted(texts))}"
            for vo, texts in vo_map.items()
        ]
        #######################################

        spans = [(m["start"], m["end"]) for m in matches]

        highlighted = highlight_text_gradient(abstract, spans)

        html_blocks.append(
            f"""
            <div class="abstract">
                <div class="meta">
                    <b>PMID:</b> {pmid}<br/>
                    <b>Title:</b> {html.escape(title)}<br/>
                    <b>Mentions:</b> {len(matches)} |
                    <b>Unique adjuvants:</b> {len(set(m["adjuvant_vo_id"] for m in matches))}<br/>
                    <b>Adjuvants (by VO):</b> {" | ".join(vo_lines)}<br/>
                </div>

                <div>{highlighted}</div>
            </div>
            <hr/>
            """
        )

html_blocks.append(HTML_FOOTER)

# -------------------------
# Save
# -------------------------
Path(OUTPUT_HTML).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_HTML, "w", encoding="utf-8") as out:
    out.write("\n".join(html_blocks))

print(f"✅ Green gradient preview saved to:\n{OUTPUT_HTML}")


✅ Green gradient preview saved to:
Dataset/VIOLIN_12-10-2025/interim/exact_matches/green_highlight_preview.html


In [2]:
""" Checking co-occurrences of pathogen for multiple VO in the main dataset
import pandas as pd

df = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv")

# Basic VO-pathogen counts
co = (
    df[["c_adjuvant_vo_id", "c_pathogen_name"]]
    .dropna()
    .value_counts()
    .reset_index(name="count")
)

# Save full table
co.to_csv("Dataset/VIOLIN_12-10-2025/interim/vo_pathogen_cooccurrence_temp.csv", index=False)

# Filter to alum-related VOs (by label/name)
alum_vos = (
    df[["c_adjuvant_vo_id", "c_adjuvant_label", "c_vaxjo_name"]]
    .dropna(subset=["c_adjuvant_vo_id"])
    .assign(name=lambda d: d["c_adjuvant_label"].fillna(d["c_vaxjo_name"]).str.lower())
)
alum_vo_ids = alum_vos[alum_vos["name"].str.contains("alum|aluminum", na=False)]["c_adjuvant_vo_id"].unique()

alum_co = co[co["c_adjuvant_vo_id"].isin(alum_vo_ids)]
alum_co.to_csv("Dataset/VIOLIN_12-10-2025/interim/alum_vo_pathogen_cooccurrence_temp.csv", index=False)

print("Alum VO IDs:", alum_vo_ids)
print(alum_co.head(20))
"""

' Checking co-occurrences of pathogen for multiple VO in the main dataset\nimport pandas as pd\n\ndf = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/infectious_df_pubmed_final.csv")\n\n# Basic VO-pathogen counts\nco = (\n    df[["c_adjuvant_vo_id", "c_pathogen_name"]]\n    .dropna()\n    .value_counts()\n    .reset_index(name="count")\n)\n\n# Save full table\nco.to_csv("Dataset/VIOLIN_12-10-2025/interim/vo_pathogen_cooccurrence_temp.csv", index=False)\n\n# Filter to alum-related VOs (by label/name)\nalum_vos = (\n    df[["c_adjuvant_vo_id", "c_adjuvant_label", "c_vaxjo_name"]]\n    .dropna(subset=["c_adjuvant_vo_id"])\n    .assign(name=lambda d: d["c_adjuvant_label"].fillna(d["c_vaxjo_name"]).str.lower())\n)\nalum_vo_ids = alum_vos[alum_vos["name"].str.contains("alum|aluminum", na=False)]["c_adjuvant_vo_id"].unique()\n\nalum_co = co[co["c_adjuvant_vo_id"].isin(alum_vo_ids)]\nalum_co.to_csv("Dataset/VIOLIN_12-10-2025/interim/alum_vo_pathogen_cooccurrence_temp.csv", index=False)\n\nprin